# Module 4 - Topic 4: Conversation History & Context Management
**AI Application Development - Developer Track (NIGCOMSAT)**

In this demo you add multi-turn conversation support to the citation pipeline from Topic 3. The LLM rewrites follow-up questions into standalone search queries, conversation history is stored as a message list, and a token budget keeps the context window under control. One concept per cell; run them in order.

**Prerequisite:** Run the Topic 2 notebook first — this notebook reuses its ChromaDB collection.

## Setup
Connect to the ChromaDB collection from Topic 2 and set up both clients (embedding + LLM).

In [1]:
import os
import re
from dotenv import load_dotenv
from openai import OpenAI
import chromadb
from chromadb import EmbeddingFunction, Documents, Embeddings

# Load API keys from .env in this folder (copy from Topic 2 if needed)
load_dotenv()

# --- Embedding client (same setup as Topics 2 and 3) ---
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
embedding_client = OpenAI(
    api_key=os.getenv("EMBEDDING_API_KEY"),
    base_url=os.getenv("EMBEDDING_BASE_URL"),
)


class CorpusEmbeddingFunction(EmbeddingFunction):
    """Wraps the embedding API so ChromaDB can call it during queries."""

    def __init__(self):
        pass

    def __call__(self, input: Documents) -> Embeddings:
        response = embedding_client.embeddings.create(
            model=EMBEDDING_MODEL, input=input
        )
        return [item.embedding for item in response.data]


# --- LLM client ---
llm_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)
MODEL = os.getenv("MODEL_NAME")

# --- Connect to Topic 2's ChromaDB collection ---
CHROMA_PATH = os.path.join("..", "..", "02-metadata-filtering-and-re-ranking", "demo", "chroma_db")
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_collection(
    name="novanet_hr", embedding_function=CorpusEmbeddingFunction()
)

print(f"Connected to collection 'novanet_hr' \u2014 {collection.count()} chunks")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"LLM model: {MODEL}")

Connected to collection 'novanet_hr' — 90 chunks
Embedding model: Qwen/Qwen3-Embedding-0.6B
LLM model: google/gemma-4-E4B-it


## Pipeline Helpers from Topics 2 and 3
We bring forward the retrieval, re-ranking, citation, and refusal functions built in the previous two topics. These are unchanged — today's work adds a layer on top.

In [2]:
# --- Retrieval helpers (Topic 2) ---

TYPE_BOOST = {"policy": 0.08, "sop": 0.05, "circular": 0.06, "faq": 0.00}


def rerank(results):
    """Reorder ChromaDB results by metadata signals."""
    ranked = []
    for chunk_id, doc, meta, dist in zip(
        results["ids"][0], results["documents"][0],
        results["metadatas"][0], results["distances"][0]
    ):
        similarity = 1 - dist
        score = similarity
        if meta["status"] == "current":
            score += 0.10
        score += TYPE_BOOST.get(meta["doc_type"], 0)
        if meta["date_issued"] >= 20250101:
            score += 0.05
        ranked.append({
            "id": chunk_id, "text": doc, "metadata": meta,
            "similarity": similarity, "reranked_score": score
        })
    ranked.sort(key=lambda x: x["reranked_score"], reverse=True)
    return ranked


def retrieve(query, n_results=5):
    """Retrieve, filter to current documents, and rerank."""
    results = collection.query(
        query_texts=[query], n_results=n_results,
        where={"status": "current"}
    )
    return rerank(results), results


# --- Citation helpers (Topic 3) ---

def format_chunk_for_llm(chunk):
    """Add a doc_id header so the LLM can cite the source."""
    meta = chunk["metadata"]
    header = f"[{meta['doc_id']}] {meta['title']}"
    return f"--- {header} ---\n{chunk['text']}"


def build_context(chunks, top_n=5):
    """Join the top N formatted chunks into one string for the LLM prompt."""
    return "\n\n".join(format_chunk_for_llm(c) for c in chunks[:top_n])


SYSTEM_PROMPT = """You are a NovaNet Technologies HR assistant.
Answer the employee's question using ONLY the documents below.
For each fact, add a [Source N] marker. After the answer, list sources like this:

Sources:
[Source 1] HR-POL-2025-006 \u2014 Annual Leave Policy, Section 3
[Source 2] HR-CIR-2025-001 \u2014 Revised Leave Entitlements

If the documents do not contain the answer, say:
\"I don't have information about that in the HR documents I can access.\"
Never invent facts or cite a document not provided above."""

DOC_ID_PATTERN = r'[A-Z]{2}-[A-Z]+-\d{4}-\d{3}'


def extract_citations(response_text):
    """Parse [Source ...] lines from the LLM response."""
    citations = {}
    n = 0
    for line in response_text.split("\n"):
        line = line.strip()
        if not re.match(r'^\[Source', line):
            continue
        doc_ids = re.findall(DOC_ID_PATTERN, line)
        if not doc_ids:
            continue
        n += 1
        doc_id = doc_ids[0]
        desc = re.split(DOC_ID_PATTERN, line)[-1]
        desc = re.sub(r'^[\s\]\[,\u2014\u2013\u2015\-]+', '', desc).strip()
        citations[f"Source {n}"] = {"doc_id": doc_id, "section": desc}
    return citations


def verify_citations(citations, retrieved_chunks):
    """Check each citation against the chunks we actually retrieved."""
    retrieved_doc_ids = set(c["metadata"]["doc_id"] for c in retrieved_chunks)
    verified = []
    for key, cit in citations.items():
        doc_id = cit["doc_id"]
        if doc_id in retrieved_doc_ids:
            verified.append({**cit, "key": key, "status": "VERIFIED"})
        else:
            verified.append({**cit, "key": key, "status": "UNVERIFIED",
                             "reason": "Document not in retrieved chunks"})
    return verified


REFUSAL_MESSAGE = (
    "I don't have information about that in the HR documents I can access. "
    "You may want to check with the HR department directly."
)


def should_refuse(chunks, citations, threshold=0.35):
    """Decide whether the system should refuse to answer."""
    if not chunks:
        return True, "No chunks retrieved"
    best_similarity = max(c["similarity"] for c in chunks)
    if best_similarity < threshold:
        return True, f"Best similarity {best_similarity:.3f} < threshold {threshold}"
    if not citations:
        return True, "LLM produced no citations"
    return False, "Answer supported"


def get_cited_answer(question, n_results=5, top_n=5, threshold=0.35):
    """Full citation pipeline: retrieve, generate, parse, verify, refuse or return."""
    chunks, raw_results = retrieve(question, n_results=n_results)
    context = build_context(chunks, top_n=top_n)
    response = llm_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Documents:\n{context}\n\nQuestion: {question}"},
        ],
        temperature=0.1,
    )
    answer = response.choices[0].message.content
    citations = extract_citations(answer)
    verified = verify_citations(citations, chunks[:top_n])
    refuse, reason = should_refuse(chunks, citations, threshold)
    return {
        "question": question,
        "answer": REFUSAL_MESSAGE if refuse else answer,
        "refused": refuse,
        "reason": reason,
        "citations": verified,
        "best_similarity": max(c["similarity"] for c in chunks) if chunks else 0,
    }


# Quick test — make sure the Topic 3 pipeline still works
test = get_cited_answer("How many days of annual leave do I get?")
print(f"Pipeline test: {'PASS' if not test['refused'] else 'FAIL'}")
print(f"Citations: {len(test['citations'])} | Similarity: {test['best_similarity']:.3f}")

Pipeline test: PASS
Citations: 2 | Similarity: 0.640


## The Follow-up Problem
The citation pipeline works for single questions. But what happens when the employee follows up? The word "that" in "Does that apply to contract staff?" refers to the previous answer, and ChromaDB has no idea what "that" means.

In [3]:
# Turn 1 works fine — a standalone question
q1 = "How many days of annual leave do I get?"
r1 = get_cited_answer(q1)
print(f"Turn 1: \"{q1}\"")
print(f"Result: {'ANSWERED' if not r1['refused'] else 'REFUSED'} | Sim: {r1['best_similarity']:.3f}")
print()

# Turn 2 fails — "that" has no referent in the search query
q2 = "Does that apply to contract staff?"
chunks_q2, _ = retrieve(q2)

print(f"Turn 2: \"{q2}\"")
print("\nChunks retrieved (wrong — nothing about leave):")
for i, c in enumerate(chunks_q2[:3], 1):
    m = c["metadata"]
    print(f"  {i}. [{m['doc_id']}] {m['title']} (sim: {c['similarity']:.3f})")

print("\n--- Problem: 'that' has no referent. ChromaDB searched for")
print("    'does that apply to contract staff' and found irrelevant chunks. ---")

Turn 1: "How many days of annual leave do I get?"
Result: ANSWERED | Sim: 0.640

Turn 2: "Does that apply to contract staff?"

Chunks retrieved (wrong — nothing about leave):
  1. [HR-POL-2025-006] Annual Leave Policy (Revised) (sim: 0.660)
  2. [HR-POL-2024-003] Remote Work Policy (sim: 0.514)
  3. [HR-POL-2024-002] Code of Conduct (sim: 0.496)

--- Problem: 'that' has no referent. ChromaDB searched for
    'does that apply to contract staff' and found irrelevant chunks. ---


## The Rewrite Prompt
We use the LLM itself to rewrite follow-ups into standalone queries. The prompt tells it to replace pronouns with the specific nouns they refer to, include key details from the conversation, and output only the rewritten query.

In [4]:
REWRITE_PROMPT = """Given a conversation between an employee and an HR assistant,
rewrite the employee's latest message as a standalone search query.

Rules:
1. Replace pronouns (it, that, those, they) with the specific nouns they refer to.
2. Include key details from the conversation (amounts, dates, document names).
3. Output ONLY the rewritten query \u2014 no explanation, no quotation marks.
4. If the message is already a standalone question, return it unchanged."""


def format_rewrite_input(history, latest_message):
    """Format conversation history + latest message for the rewriter."""
    lines = ["Conversation so far:"]
    for msg in history:
        role = "Employee" if msg["role"] == "user" else "Assistant"
        lines.append(f"{role}: {msg['content']}")
    lines.append(f"\nLatest message to rewrite:\n{latest_message}")
    return "\n".join(lines)


def rewrite_query(history, follow_up):
    """Rewrite a follow-up into a standalone search query using the LLM."""
    rewrite_input = format_rewrite_input(history, follow_up)
    response = llm_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": REWRITE_PROMPT},
            {"role": "user", "content": rewrite_input},
        ],
        temperature=0.0,  # No creativity — produce the most literal rewrite
    )
    return response.choices[0].message.content.strip()


print("Rewrite functions defined.")
print(f"\nRewrite prompt:\n{REWRITE_PROMPT}")

Rewrite functions defined.

Rewrite prompt:
Given a conversation between an employee and an HR assistant,
rewrite the employee's latest message as a standalone search query.

Rules:
1. Replace pronouns (it, that, those, they) with the specific nouns they refer to.
2. Include key details from the conversation (amounts, dates, document names).
3. Output ONLY the rewritten query — no explanation, no quotation marks.
4. If the message is already a standalone question, return it unchanged.


## Testing the Rewriter
We simulate a conversation about annual leave and test three follow-ups. Each should be rewritten into a standalone query that ChromaDB can search effectively.

In [5]:
# Simulate the history from Turn 1
# We strip citations from the assistant's answer to keep history clean
test_history = [
    {"role": "user", "content": "How many days of annual leave do I get?"},
    {"role": "assistant", "content": (
        "All confirmed employees are entitled to 25 working days "
        "of paid annual leave per calendar year."
    )},
]

# Three follow-ups to test
follow_ups = [
    "Does that apply to contract staff?",
    "Can I carry those over to next year?",
    "What if I don't use them all?",
]

print("History:")
for msg in test_history:
    role = "Employee" if msg["role"] == "user" else "Assistant"
    print(f"  {role}: {msg['content'][:80]}..." if len(msg['content']) > 80 else f"  {role}: {msg['content']}")

print("\nRewrite results:")
print("-" * 80)
for follow_up in follow_ups:
    rewritten = rewrite_query(test_history, follow_up)
    print(f"  Original:  {follow_up}")
    print(f"  Rewritten: {rewritten}")
    print()

History:
  Employee: How many days of annual leave do I get?
  Assistant: All confirmed employees are entitled to 25 working days of paid annual leave per...

Rewrite results:
--------------------------------------------------------------------------------
  Original:  Does that apply to contract staff?
  Rewritten: Does the 25 working days of paid annual leave per calendar year apply to contract staff?

  Original:  Can I carry those over to next year?
  Rewritten: Can I carry over 25 working days of paid annual leave to next year?

  Original:  What if I don't use them all?
  Rewritten: What happens to unused annual leave days at the end of the calendar year?



## Rewrite Fixes the Retrieval
Now let us compare: the raw follow-up vs. the rewritten query. The rewritten version should retrieve relevant chunks about leave entitlements.

In [6]:
follow_up = "Does that apply to contract staff?"
rewritten = rewrite_query(test_history, follow_up)

# Retrieval with the raw follow-up (broken)
chunks_raw, _ = retrieve(follow_up)
print(f"RAW query: \"{follow_up}\"")
for i, c in enumerate(chunks_raw[:3], 1):
    m = c["metadata"]
    print(f"  {i}. [{m['doc_id']}] {m['title']} (sim: {c['similarity']:.3f})")

print()

# Retrieval with the rewritten query (fixed)
chunks_rw, _ = retrieve(rewritten)
print(f"REWRITTEN query: \"{rewritten}\"")
for i, c in enumerate(chunks_rw[:3], 1):
    m = c["metadata"]
    print(f"  {i}. [{m['doc_id']}] {m['title']} (sim: {c['similarity']:.3f})")

print("\n--- The rewritten query finds leave-related chunks. ---")

RAW query: "Does that apply to contract staff?"
  1. [HR-POL-2025-006] Annual Leave Policy (Revised) (sim: 0.660)
  2. [HR-POL-2024-003] Remote Work Policy (sim: 0.514)
  3. [HR-POL-2024-002] Code of Conduct (sim: 0.496)

REWRITTEN query: "Does the 25 working days of paid annual leave per calendar year apply to contract staff?"
  1. [HR-POL-2025-006] Annual Leave Policy (Revised) (sim: 0.725)
  2. [HR-POL-2025-006] Annual Leave Policy (Revised) (sim: 0.648)
  3. [HR-POL-2025-006] Annual Leave Policy (Revised) (sim: 0.605)

--- The rewritten query finds leave-related chunks. ---


## Conversation History Storage
History is a list of `{role, content}` dictionaries — the same format the OpenAI API uses. We keep a sliding window of the last N exchanges to avoid sending the entire conversation to the LLM.

In [7]:
def get_history_window(history, max_pairs=5):
    """Return the last N user-assistant pairs from history."""
    # Each pair is 2 messages (user + assistant)
    max_messages = max_pairs * 2
    return history[-max_messages:]


def strip_citations(answer_text):
    """Remove [Source N] markers and the Sources block from an answer.

    The rewriter only needs the plain answer — citation markers are noise.
    """
    # Remove inline [Source N] markers
    text = re.sub(r'\[Source\s*\d*\]', '', answer_text)
    # Remove the Sources block at the end
    text = re.split(r'\n\s*Sources?:', text)[0]
    # Clean up extra whitespace
    text = re.sub(r'  +', ' ', text).strip()
    return text


# Test strip_citations
sample = (
    "Employees get 25 working days [Source 1] of annual leave. "
    "Carry-over is limited to 10 days [Source 2].\n\n"
    "Sources:\n[Source 1] HR-POL-2025-006 — Annual Leave Policy\n"
    "[Source 2] HR-POL-2025-006 — Section 3c"
)
print("Before stripping:")
print(f"  {sample[:100]}...")
print(f"\nAfter stripping:")
print(f"  {strip_citations(sample)}")

# Test history window
long_history = [{"role": "user", "content": f"Q{i}"} for i in range(1, 11)]
window = get_history_window(long_history, max_pairs=3)
print(f"\nHistory window: {len(long_history)} messages → last {len(window)} kept")

Before stripping:
  Employees get 25 working days [Source 1] of annual leave. Carry-over is limited to 10 days [Source 2...

After stripping:
  Employees get 25 working days of annual leave. Carry-over is limited to 10 days .

History window: 10 messages → last 6 kept


## Token Budget and Trimming
The LLM's context window has a fixed size. We estimate token counts with a simple heuristic (1 token ≈ 4 characters) and trim the oldest messages when history exceeds the budget.

In [8]:
def estimate_tokens(text):
    """Rough estimate: 1 token \u2248 4 characters."""
    return len(text) // 4


def trim_history(history, max_tokens=5000):
    """Remove oldest messages until history fits the token budget."""
    trimmed = list(history)
    while trimmed and estimate_tokens(str(trimmed)) > max_tokens:
        trimmed.pop(0)  # Drop the oldest message
    return trimmed


# Demonstrate: build a long history and trim it
long_history = []
for i in range(1, 21):
    long_history.append({"role": "user", "content": f"Question number {i} about HR policies at NovaNet Technologies?"})
    long_history.append({"role": "assistant", "content": f"Answer {i}: Here is a detailed response about the policy " * 5})

tokens_before = estimate_tokens(str(long_history))
trimmed = trim_history(long_history, max_tokens=2000)
tokens_after = estimate_tokens(str(trimmed))

print(f"Before trimming: {len(long_history)} messages, ~{tokens_before} tokens")
print(f"After trimming:  {len(trimmed)} messages, ~{tokens_after} tokens")
print(f"Dropped {len(long_history) - len(trimmed)} oldest messages to fit budget")

Before trimming: 40 messages, ~2046 tokens
After trimming:  38 messages, ~1945 tokens
Dropped 2 oldest messages to fit budget


## The Full Multi-turn Pipeline
One function that handles both first questions and follow-ups: check if history exists, rewrite if needed, call the citation pipeline, and append the exchange to history.

In [9]:
def chat(message, history, max_pairs=5, max_history_tokens=5000):
    """Full multi-turn pipeline: rewrite \u2192 retrieve \u2192 cite \u2192 append."""

    # Step 1: Decide whether to rewrite
    if history:
        # Trim history to token budget before sending to the rewriter
        window = get_history_window(history, max_pairs=max_pairs)
        window = trim_history(window, max_tokens=max_history_tokens)
        search_query = rewrite_query(window, message)
    else:
        # First question — no history, no rewrite needed
        search_query = message

    # Step 2: Run the citation pipeline with the (possibly rewritten) query
    result = get_cited_answer(search_query)

    # Step 3: Append this exchange to history
    # Store the original message (what the employee typed)
    history.append({"role": "user", "content": message})
    # Store the answer without citations (cleaner for the rewriter)
    clean_answer = strip_citations(result["answer"])
    history.append({"role": "assistant", "content": clean_answer})

    # Add the rewritten query to the result for logging
    result["original_message"] = message
    result["search_query"] = search_query
    result["was_rewritten"] = (search_query != message)

    return result


print("chat() function defined — ready for multi-turn conversations.")

chat() function defined — ready for multi-turn conversations.


## Running a Multi-turn Conversation
A three-turn conversation: the employee asks about annual leave, follows up about carry-over, then asks about the deadline. Each turn builds on the previous one.

In [10]:
# Start a fresh conversation
conversation_history = []

turns = [
    "How many days of annual leave do I get?",
    "How many of those can I carry over to next year?",
    "What is the deadline for using them?",
]

for i, message in enumerate(turns, 1):
    print(f"{'='*80}")
    print(f"TURN {i}")
    print(f"Employee: {message}")

    result = chat(message, conversation_history)

    # Show the rewrite if it happened
    if result["was_rewritten"]:
        print(f"Rewritten: {result['search_query']}")
    else:
        print(f"Search query: {result['search_query']} (no rewrite needed)")

    # Show the answer (truncated for readability)
    answer_preview = result["answer"][:200]
    if len(result["answer"]) > 200:
        answer_preview += "..."
    print(f"\nAssistant: {answer_preview}")

    # Show metadata
    status = "REFUSED" if result["refused"] else "ANSWERED"
    n_cites = len(result["citations"])
    print(f"\nStatus: {status} | Citations: {n_cites} | Sim: {result['best_similarity']:.3f}")
    print(f"History length: {len(conversation_history)} messages")
    print()

TURN 1
Employee: How many days of annual leave do I get?
Search query: How many days of annual leave do I get? (no rewrite needed)

Assistant: Confirmed full-time employees are entitled to 25 working days of paid annual leave per calendar year [Source 1, Source 3]. Employees in their first year of employment receive a pro-rated entitlement c...

Status: ANSWERED | Citations: 2 | Sim: 0.640
History length: 2 messages

TURN 2
Employee: How many of those can I carry over to next year?
Rewritten: How many days of annual leave can I carry over to next year?

Assistant: You may carry forward a maximum of 10 unused leave days to the next calendar year [Source 1, Source 2, Source 3]. These carried-forward days must be used by 31 March of the following year, or they wil...

Status: ANSWERED | Citations: 3 | Sim: 0.640
History length: 4 messages

TURN 3
Employee: What is the deadline for using them?
Rewritten: What is the deadline for using carried-forward annual leave days?

Assistant: Carried-

## Edge Cases: Topic Switches and Greetings
Not every message is a follow-up. A standalone question should pass through unchanged. A greeting or thanks should not trigger retrieval at all.

In [11]:
# Continue the same conversation — test a topic switch
print("--- Topic Switch ---")
switch_msg = "What is the remote work allowance?"
result = chat(switch_msg, conversation_history)
print(f"Employee: {switch_msg}")
if result["was_rewritten"]:
    print(f"Rewritten: {result['search_query']}")
else:
    print(f"Search query: {result['search_query']} (no rewrite — already standalone)")
status = "REFUSED" if result["refused"] else "ANSWERED"
print(f"Status: {status} | Citations: {len(result['citations'])} | Sim: {result['best_similarity']:.3f}")

print()

# Test a follow-up on the NEW topic
print("--- Follow-up on New Topic ---")
followup_msg = "Was it always that amount?"
result2 = chat(followup_msg, conversation_history)
print(f"Employee: {followup_msg}")
if result2["was_rewritten"]:
    print(f"Rewritten: {result2['search_query']}")
print(f"Status: {'REFUSED' if result2['refused'] else 'ANSWERED'} | Citations: {len(result2['citations'])} | Sim: {result2['best_similarity']:.3f}")

print()

# Test a refusal question as a follow-up
print("--- Off-topic Question ---")
offtopic_msg = "Can I bring my pet to the office?"
result3 = chat(offtopic_msg, conversation_history)
print(f"Employee: {offtopic_msg}")
if result3["was_rewritten"]:
    print(f"Rewritten: {result3['search_query']}")
print(f"Status: {'REFUSED' if result3['refused'] else 'ANSWERED'} | Sim: {result3['best_similarity']:.3f}")
print(f"\nHistory length: {len(conversation_history)} messages (6 turns total)")

--- Topic Switch ---
Employee: What is the remote work allowance?
Search query: What is the remote work allowance? (no rewrite — already standalone)
Status: ANSWERED | Citations: 3 | Sim: 0.689

--- Follow-up on New Topic ---
Employee: Was it always that amount?
Rewritten: Was the monthly Remote Work Allowance always NGN 35,000?
Status: ANSWERED | Citations: 2 | Sim: 0.718

--- Off-topic Question ---
Employee: Can I bring my pet to the office?
Status: REFUSED | Sim: 0.459

History length: 12 messages (6 turns total)


## Recap

- **The follow-up problem:** raw follow-ups fail at retrieval because pronouns like "that" have no referent in the search query
- **Query rewriting:** the LLM rewrites follow-ups into standalone queries before they hit ChromaDB
- **Rewrite prompt:** replace pronouns, include key details, output only the query, skip if already standalone
- **History storage:** a list of `{role, content}` dictionaries, with citations stripped for cleaner context
- **Token budget:** rough estimate (1 token ≈ 4 chars), trim oldest messages first
- **`chat()` function:** ties it all together — rewrite → retrieve → cite → append to history

This becomes `history.py` in the reference application. Next topic — **Building the Streamlit Interface** — puts a chat UI on top of everything, so a non-technical employee can use the assistant without touching Python.